# Benchmarking `GMP(lam)` vs `magnus_exp(lam)`

This notebook benchmarks the two independent algorithms available in the library for computing Generalized Macdonald functions (GMPs):

- **`GMP(lam)`** -- the *kernel method*: finds $P_{\boldsymbol\lambda}$ as the null vector of the eigenvalue-equation matrix, via `GMPC`. Results are memoized in the global `GMPC_cache`.
- **`magnus_exp(lam)`** -- the *Magnus expansion*: an iterative refinement starting from the pure tensor $P_{\lambda^{(0)}} \otimes \cdots \otimes P_{\lambda^{(N-1)}}$, converging in finitely many steps. **Not cached.**

Both algorithms should always agree on the result; this notebook checks that too, alongside the timings.

To make this a fair comparison, the benchmark below times `GMP` **twice** per test case:

- **cold** -- immediately after wiping `GMPC_cache` completely
- **warm** -- immediately after that, with the cache now populated

and reports both, so the caching benefit is visible. `magnus_exp` is also timed twice in a row.

## Setup

Load the library.

In [1]:
load("../gmplib.py")
import time

## The benchmark function

`benchmark_kernel_vs_magnus` runs both algorithms on a list of multi-partitions and records four timings per case: `magnus_exp` called twice in a row, and `GMP` called cold-then-warm. It also checks that all four results agree with each other.

In [2]:
def benchmark_kernel_vs_magnus(cases, warm_up_reps=1):
    """
    Compare GMP(lam) [kernel method, cached] against magnus_exp(lam)
    [Magnus expansion, uncached] across a list of multi-partitions.

    See the notebook markdown above for why GMP is timed cold AND warm,
    and why the pre-existing cache is saved/restored around the run.

    Parameters
    ----------
    cases : list of tuple-of-partitions
        Multi-partitions to benchmark, e.g. from mPartitions(N, d).
    warm_up_reps : int
        Number of GMP warm-call repetitions to average over (default 1).

    Returns
    -------
    list of dict
        One dict per case with timing and correctness results.
    """
    global GMPC_cache
    saved_cache = dict(GMPC_cache)  # snapshot: values are immutable ring elements

    # settle Sage's own lower-level lazy caches before real measurement
    _ = GMP(([1],))
    _ = magnus_exp(([1],))
    gmp_init()

    results = []
    try:
        for lam in cases:
            lam = tuple(Partition(p) for p in lam)
            N = len(lam)
            degree = sum(map(sum, lam))

            # ---- magnus_exp: called twice, no caching expected either time ----
            t0 = time.perf_counter()
            G_magnus = magnus_exp(lam)
            t_magnus_1 = time.perf_counter() - t0

            t0 = time.perf_counter()
            G_magnus_2 = magnus_exp(lam)
            t_magnus_2 = time.perf_counter() - t0

            # ---- GMP, cold: wipe the ENTIRE cache first (no per-lam clear exists) ----
            gmp_init()
            t0 = time.perf_counter()
            G_gmp_cold = GMP(lam)
            t_gmp_cold = time.perf_counter() - t0

            # ---- GMP, warm: cache now populated from the cold call above ----
            t0 = time.perf_counter()
            for _ in range(warm_up_reps):
                G_gmp_warm = GMP(lam)
            t_gmp_warm = (time.perf_counter() - t0) / warm_up_reps

            match = (G_magnus == G_magnus_2 == G_gmp_cold == G_gmp_warm)

            results.append({
                'lam': lam, 'N': N, 'degree': degree,
                't_magnus_1': t_magnus_1, 't_magnus_2': t_magnus_2,
                't_gmp_cold': t_gmp_cold, 't_gmp_warm': t_gmp_warm,
                'match': match,
            })
    finally:
        gmp_init(saved_cache)  # always restore, even if something above raised

    return results

## Pretty-printing the results

In [3]:
def print_benchmark_table(results):
    r"""Pretty-print the results of benchmark_kernel_vs_magnus."""
    header = (f"{'N':>2} {'deg':>4} {'lam':<26} {'match':>7} "
              f"{'magnus_1':>10} {'magnus_2':>10} "
              f"{'GMP cold':>10} {'GMP warm':>10} {'speedup×':>8}")
    print(header)
    print("-" * len(header))
    all_ok = True
    avrg_speedup = 0
    for r in results:
        all_ok = all_ok and r['match']
        speedup = (r['t_magnus_1'] + r['t_magnus_2']) / 2 / r['t_gmp_cold']
        avrg_speedup += speedup
        lam_str = str(r['lam'])
        print(f"{r['N']:>2} {r['degree']:>4} {lam_str:<26} "
              f"{'OK' if r['match'] else 'FAIL':>7} "
              f"{r['t_magnus_1']*1000:>9.3f}m "
              f"{r['t_magnus_2']*1000:>9.3f}m "
              f"{r['t_gmp_cold']*1000:>9.3f}m "
              f"{r['t_gmp_warm']*1000:>9.3f}m "
              f"{speedup:>7.1f}×")
    print("-" * len(header))
    avrg_speedup = avrg_speedup / len(results)
    print(f"All results matched.\nAverage speedup of kernel method compared to Magnus: {avrg_speedup:>7.1f}x" if all_ok else "MISMATCHES DETECTED -- see FAIL rows above.")

## Running it

The `match` column should read `OK` for every row.

In [ ]:
test_cases = [lam for d in range(5) for N in range(1, 4) for lam in mPartitions(N, d)]

results = benchmark_kernel_vs_magnus(test_cases)
print_benchmark_table(results)

- **The `speedup×` column** compares the kernel method (cold) against the Magnus expansion method (average between the two runs).
- **`magnus_1` vs `magnus_2`** should land close together on every row.